# 03 — Reference

Builds the community-area lookup table used for the equity analysis: name, per-capita income, poverty rate, and hardship index for each of Chicago's 77 community areas. Pulled from the city's [socioeconomic indicators dataset](https://data.cityofchicago.org/Health-Human-Services/Census-Data-Selected-socioeconomic-indicators-in-C/kn9c-c2s2) (`kn9c-c2s2`) and saved to `data/community_lookup.parquet`.

In [ ]:
import pandas as pd
from sodapy import Socrata

client = Socrata("data.cityofchicago.org", None, timeout=60)

socio = client.get("kn9c-c2s2", limit=100)
socio_df = pd.DataFrame.from_records(socio)
print(socio_df.columns.tolist())
print(f"{len(socio_df)} rows")
socio_df.head()

## Select and rename

Keep only the columns the analysis uses, rename them for clarity, and drop the citywide total row (it has no community-area number).

In [ ]:
# keep the columns we'll actually use
lookup = socio_df[[
    "ca",
    "community_area_name",
    "percent_households_below_poverty",
    "per_capita_income_",
    "hardship_index"
]].copy()

# rename for clarity
lookup = lookup.rename(columns={
    "ca": "community_area",
    "community_area_name": "area_name",
    "percent_households_below_poverty": "pct_below_poverty",
    "per_capita_income_": "per_capita_income",
    "hardship_index": "hardship_index"
})

# cast types
lookup["community_area"] = pd.to_numeric(lookup["community_area"], errors="coerce")
for col in ["pct_below_poverty", "per_capita_income", "hardship_index"]:
    lookup[col] = pd.to_numeric(lookup[col], errors="coerce")

# drop the citywide total row (it has no community_area number)
lookup = lookup.dropna(subset=["community_area"])
lookup["community_area"] = lookup["community_area"].astype(int)

print(f"{len(lookup)} community areas")  # should be 77
print(lookup.dtypes)
lookup.head()

## Save

In [ ]:
import os
lookup.to_parquet("data/community_lookup.parquet", index=False)
print(f"Saved. {os.listdir('data')}")